In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd


import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:

config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last.yaml"
# config_path = "config/binaural_attn/word_task_half_co_loc_v06.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 8
config['audio']['rep_kwargs']['rep_on_gpu'] = True
config['model']['norm_first'] = False
# print(config['hparas']['lr'])
# config


In [3]:
seed_everything(0)
import src.spatial_attn_lightning as binaural_lightning 
config['model']['norm_first'] = True

importlib.reload(binaural_lightning)
module = binaural_lightning.BinauralAttentionModule(config)
print(module.model.output_height, module.model.output_len, module.model.output_size)
for name, param in module.model.named_parameters():
    print(name, param.numel())


[rank: 0] Seed set to 0


Using explicit dim specificaion for demeaning in audio transforms
num_classes={'num_words': 800}
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
6 16 49152
_orig_mod.model_dict.norm_coch_rep.weight 1600000
_orig_mod.model_dict.norm_coch_rep.bias 1600000
_orig_mod.model_dict.attn_block_in.bias 1
_orig_mod.model_dict.attn_block_in.slope 1
_orig_mod.model_dict.attn_block_in.threshold 1
_orig_mod.model_dict.conv_block_0.0.weight 1600000
_orig_mod.model_dict.conv_block_0.0.bias 1600000
_orig_mod.model_dict.conv_block_0.1.weight 4352
_orig_mod.model_dict.attn0.bias 1
_orig_mod.model_dict.attn0.slope 1
_orig_mod.model_dict.attn0.threshold 1
_orig_mod.model_dict.conv_block_1.0.weight 3200000
_orig_mod.model_dict.conv_block_1.0.bias 3200000
_orig_mod.model_dict.conv_block_1.1.weight 57344
_orig_mod.model_dict.attn1.bias 1
_orig_mod.model_dict.attn1.slope 1
_orig_mod.model_dict.attn1.threshold 1
_orig_mod.model_dict.conv_block_2.0.weight 800000
_orig_mod.model_dict.conv

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


#### Putting layer-norm last increases number of trainable parameters at each layer:
    [channels_in x height x width] -> [channels_out x height x width]

In [4]:
# seed_everything(0)
# import src.spatial_attn_lightning as binaural_lightning 

# importlib.reload(binaural_lightning)
# config['model']['norm_first'] = False
# module = binaural_lightning.BinauralAttentionModule(config)
# # get n parameters per layer
# for name, param in module.model.named_parameters():
#     print(name, param.numel())


In [5]:
seed_everything(0)
import src.spatial_attn_lightning as binaural_lightning 

importlib.reload(binaural_lightning)
config['model']['norm_first'] = False
config['model']['ln_affine'] = False
# change padding type to valid-time
config['model']['padding'] = ['valid_time' for _ in config['model']['padding']]

module = binaural_lightning.BinauralAttentionModule(config)
print(module.model.output_height, module.model.output_len, module.model.output_size)
# 6 16 49152
# get n parameters per layer
for name, param in module.model.named_parameters():
    print(name, param.numel())


[rank: 0] Seed set to 0


Using explicit dim specificaion for demeaning in audio transforms
num_classes={'num_words': 800}
Model performing word task
Conv block order: Conv -> ReLU -> LN
nIn: 2, nOut: 32, kernel: [2, 34], stride: [1, 1], padding: valid_time
output height: 39, output len: 19967
nIn: 32, nOut: 64, kernel: [2, 14], stride: [1, 1], padding: valid_time
output height: 19, output len: 4979
nIn: 64, nOut: 256, kernel: [5, 5], stride: [1, 1], padding: valid_time
output height: 10, output len: 1241
nIn: 256, nOut: 512, kernel: [5, 5], stride: [1, 1], padding: valid_time
output height: 10, output len: 245


nIn: 512, nOut: 512, kernel: [6, 6], stride: [1, 1], padding: valid_time
output height: 9, output len: 57
nIn: 512, nOut: 512, kernel: [5, 5], stride: [1, 1], padding: valid_time
output height: 9, output len: 53
nIn: 512, nOut: 512, kernel: [6, 6], stride: [1, 1], padding: valid_time
output height: 8, output len: 48
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
5 12 30720
_orig_mod.model_dict.norm_coch_rep.weight 1600000
_orig_mod.model_dict.norm_coch_rep.bias 1600000
_orig_mod.model_dict.attn0.bias 1
_orig_mod.model_dict.attn0.slope 1
_orig_mod.model_dict.attn0.threshold 1
_orig_mod.model_dict.conv_block_0.0.weight 4352
_orig_mod.model_dict.attn1.bias 1
_orig_mod.model_dict.attn1.slope 1
_orig_mod.model_dict.attn1.threshold 1
_orig_mod.model_dict.conv_block_1.0.weight 57344
_orig_mod.model_dict.attn2.bias 1
_orig_mod.model_dict.attn2.slope 1
_orig_mod.model_dict.attn2.threshold 1
_orig_mod.model_dict.conv_block_2.0.weight 409600
_orig_mod.model_dict.attn3.b

In [6]:
config['model']

{'input_sr': 10000,
 'out_channels': [32, 64, 256, 512, 512, 512, 512],
 'kernel': [[2, 34], [2, 14], [5, 5], [5, 5], [6, 6], [5, 5], [6, 6]],
 'stride': [[1, 1], [1, 1], [1, 1], [1, 1], [1, 1], [1, 1], [1, 1]],
 'padding': ['valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time'],
 'pool_stride': [[2, 4], [2, 4], [1, 5], [1, 4], [1, 1], [1, 1], [2, 4]],
 'pool_size': [[9, 13], [9, 13], [1, 13], [1, 13], [1, 1], [1, 1], [6, 13]],
 'pool_padding': [[4, 6], [4, 6], [0, 6], [0, 6], [0, 0], [0, 0], [3, 6]],
 'attn': [1, 1, 1, 1, 1, 1, 1],
 'num_classes': {'num_words': 800},
 'fc_size': 512,
 'global_avg_cue': False,
 'dropout': 0.6,
 'attn_constraints': {'slope': True},
 'norm_first': False,
 'ln_affine': False}

In [7]:
# ## Setup models 
# import src.spatial_attn_architecture as binaural_arch
# importlib.reload(binaural_arch)
# BinauralAuditoryAttentionCNN = binaural_arch.BinauralAuditoryAttentionCNN 

# model = BinauralAuditoryAttentionCNN(**config['model'])

# cochlea = module.coch_gram
# # model = torch.compile(model)

## Make sure single example audio transforms do same thing when batched 

In [8]:
# # Make sure _collate function doesn't do something strange
# config['hparas']['batch_size'] = 16
# train_dataset = module.dataset(**module.corpora_config,
#                                 batch_size=config['hparas']['batch_size'] ,
#                                 mode='train')
# train_dataloader = module.train_dataloader()

In [9]:
from tqdm import tqdm

In [10]:
# model = model.cuda()
# # coch_gram = module.coch_gram.cuda()
# # 
# for ix, batch in enumerate(tqdm(train_dataloader, total=100)):
#     cue_features, cue_mask_ixs, scene_features, labels = batch
#     # cue_features, scene_features = coch_gram(cue_features.cuda(), scene_features.cuda())
#     preds = model(cue_features, scene_features, cue_mask_ixs)
#     if ix == 100:
#         break


In [11]:
trainer = Trainer(
    precision="32",
    # limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model            | OptimizedModule      

Sanity Checking: |          | 0/? [00:00<?, ?it/s]Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v07
cue type: mixed
mixture_percentages={'voice_only': 0.25, 'voice_and_location': 0.75}
49 files in val concat dataset
Using v06 dataset                                                          
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v07
cue type: mixed
mixture_percentages={'voice_only': 0.25, 'voice_and_location': 0.75}
977 files in train concat dataset
len training set = 488500
Epoch 0:   0%|          | 71/488500 [02:29<286:36:18,  0.47it/s, v_num=3.55e+7, train_loss=6.900, grad_norm=24.00]

In [ ]:
## Check dataset 

dataset = module.dataset(**config['corpus'], mode='train')

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v07
cue type: mixed
mixture_percentages={'voice_only': 0.25, 'voice_and_location': 0.75}
977 files in train concat dataset


In [ ]:
cue, target, distractor, labels = dataset[0]

In [ ]:
aud_transforms = module.audio_transforms

In [ ]:
ix = 1080

cue, target, distractor, labels = dataset[ix]

scene, _ = aud_transforms(target, distractor)

print("cue")
ipd.display(ipd.Audio(cue[0], rate=44100, normalize=False))
print("target")
ipd.display(ipd.Audio(target[0], rate=44100, normalize=False))
print("distractor")
ipd.display(ipd.Audio(distractor[0], rate=44100, normalize=False))
print("scene")
ipd.display(ipd.Audio(scene[0], rate=44100, normalize=False))

cue


target


distractor


IndexError: index 0 is out of bounds for axis 0 with size 0